# Hydro Temporal GNN — Kaggle launcher

Notebook ini hanya menemukan/menyiapkan project dan memanggil entry point modular. Seluruh preprocessing, graph construction, training, evaluasi, inference, dan submission tetap berada di `src/` dan `scripts/`.

In [ ]:
import platform
import sys
from pathlib import Path

print(f'Python: {sys.version}')
print(f'Platform: {platform.platform()}')
print(f'Working directory: {Path.cwd()}')
print(f'Kaggle working exists: {Path("/kaggle/working").exists()}')

In [ ]:
import shutil
import subprocess

WORKING_REPOSITORY = Path('/kaggle/working/graph-nn-tma-mdpl')
PROJECT_ROOT = WORKING_REPOSITORY / 'notebook-graph-modular'

if not PROJECT_ROOT.is_dir():
    uploaded_candidates = sorted(Path('/kaggle/input').glob('**/notebook-graph-modular'))
    if uploaded_candidates:
        source_project = uploaded_candidates[0]
        WORKING_REPOSITORY.mkdir(parents=True, exist_ok=True)
        shutil.copytree(
            source_project,
            PROJECT_ROOT,
            ignore=shutil.ignore_patterns('__pycache__', '*.pyc', 'outputs'),
        )
        print(f'Copied uploaded project: {source_project} -> {PROJECT_ROOT}')
    else:
        clone_command = [
            'git', 'clone', '--branch', 'modular-version',
            'https://github.com/ibrahim119920/graph-nn-tma-mdpl.git',
            str(WORKING_REPOSITORY),
        ]
        try:
            subprocess.run(clone_command, check=True)
        except subprocess.CalledProcessError as error:
            raise FileNotFoundError(
                'notebook-graph-modular tidak ditemukan. Upload project sebagai Kaggle Dataset '
                'atau aktifkan Internet lalu jalankan git clone branch modular-version. '
                f'Perintah gagal: {clone_command}'
            ) from error

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f'Project root tidak tersedia setelah bootstrap: {PROJECT_ROOT}')

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'kaggle.json'
print(f'Project root: {PROJECT_ROOT}')
print(f'Config: {CONFIG_PATH}')

In [ ]:
DATASET_ROOT = Path('/kaggle/input/competitions/sebelas-maret-statistics-data-science-2026')
SAMPLE_SUBMISSION = Path('/kaggle/input/competitions/sebelas-maret-statistics-data-science-2026/sample_submission.csv')

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f'Kaggle dataset root tidak ditemukan: {DATASET_ROOT}')
if not SAMPLE_SUBMISSION.is_file():
    raise FileNotFoundError(f'sample_submission.csv tidak ditemukan: {SAMPLE_SUBMISSION}')
print(f'Dataset root OK: {DATASET_ROOT}')
print(f'Sample submission OK: {SAMPLE_SUBMISSION}')

In [ ]:
subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'scripts' / 'smoke_test.py')],
    check=True,
)

In [ ]:
subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'run_pipeline.py'),
        '--config', str(CONFIG_PATH),
    ],
    check=True,
)

In [ ]:
import pandas as pd

OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
SUBMISSION_PATH = OUTPUT_ROOT / 'submissions' / 'submission.csv'
if not SUBMISSION_PATH.is_file():
    raise FileNotFoundError(f'Pipeline selesai tetapi submission tidak ditemukan: {SUBMISSION_PATH}')
submission = pd.read_csv(SUBMISSION_PATH)
print(f'Output root: {OUTPUT_ROOT}')
print(f'Submission: {SUBMISSION_PATH}')
print(f'Submission shape: {submission.shape}')
display(submission.head())

In [ ]:
# ============================================================
# CODEX HANDOFF EXPORT
# Membuat ZIP hasil eksperimen dengan timestamp Asia/Jakarta
# ============================================================

%cd /kaggle/working/graph-nn-tma-mdpl/notebook-graph-modular

import json
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import torch


# ============================================================
# 1. Path dan timestamp
# ============================================================

PROJECT_ROOT = Path.cwd()
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
HANDOFF_METADATA_DIR = OUTPUT_ROOT / "handoff"

HANDOFF_METADATA_DIR.mkdir(parents=True, exist_ok=True)

jakarta_now = datetime.now(ZoneInfo("Asia/Jakarta"))
timestamp = jakarta_now.strftime("%Y%m%d_%H%M%S")

handoff_name = f"codex-handoff_{timestamp}.zip"
handoff_path = Path("/kaggle/working") / handoff_name

print(f"Project root : {PROJECT_ROOT}")
print(f"Waktu Jakarta: {jakarta_now.isoformat()}")
print(f"Output ZIP   : {handoff_path}")


# ============================================================
# 2. Helper untuk menjalankan command
# ============================================================

def run_command(command: list[str]) -> str:
    try:
        result = subprocess.run(
            command,
            cwd=PROJECT_ROOT,
            capture_output=True,
            text=True,
            check=True,
        )
        return result.stdout.strip()
    except Exception as exc:
        return f"Unavailable: {type(exc).__name__}: {exc}"


# ============================================================
# 3. Simpan metadata environment dan Git
# ============================================================

metadata = {
    "handoff_name": handoff_name,
    "created_at_jakarta": jakarta_now.isoformat(),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "python_version": sys.version,
    "platform": platform.platform(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "cudnn_version": (
        torch.backends.cudnn.version()
        if torch.backends.cudnn.is_available()
        else None
    ),
    "gpu_count": torch.cuda.device_count(),
    "gpu_names": [
        torch.cuda.get_device_name(index)
        for index in range(torch.cuda.device_count())
    ],
    "git_branch": run_command(["git", "branch", "--show-current"]),
    "git_commit": run_command(["git", "rev-parse", "HEAD"]),
    "git_status": run_command(["git", "status", "--short"]),
}

environment_path = HANDOFF_METADATA_DIR / "environment.json"

with environment_path.open("w", encoding="utf-8") as file:
    json.dump(
        metadata,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(f"Metadata tersimpan: {environment_path}")


# ============================================================
# 4. Simpan daftar package
# ============================================================

pip_freeze_path = HANDOFF_METADATA_DIR / "pip-freeze.txt"

with pip_freeze_path.open("w", encoding="utf-8") as file:
    file.write(
        run_command([
            sys.executable,
            "-m",
            "pip",
            "freeze",
        ])
    )

print(f"Package list tersimpan: {pip_freeze_path}")


# ============================================================
# 5. Membuat inventory seluruh artifact
# ============================================================

inventory = []

if OUTPUT_ROOT.exists():
    for path in sorted(OUTPUT_ROOT.rglob("*")):
        if not path.is_file():
            continue

        stat = path.stat()

        inventory.append({
            "path": str(path.relative_to(PROJECT_ROOT)),
            "size_bytes": stat.st_size,
            "modified_at_utc": datetime.fromtimestamp(
                stat.st_mtime,
                tz=timezone.utc,
            ).isoformat(),
        })

inventory_path = HANDOFF_METADATA_DIR / "artifact-inventory.json"

with inventory_path.open("w", encoding="utf-8") as file:
    json.dump(
        inventory,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(f"Artifact terdaftar: {len(inventory)} file")
print(f"Inventory tersimpan: {inventory_path}")


# ============================================================
# 6. Tentukan folder yang akan dimasukkan ke ZIP
# ============================================================

handoff_candidates = [
    Path("outputs/experiments"),
    Path("outputs/logs"),
    Path("outputs/submissions"),
    Path("outputs/handoff"),
]

existing_paths = [
    path
    for path in handoff_candidates
    if (PROJECT_ROOT / path).exists()
]

missing_paths = [
    str(path)
    for path in handoff_candidates
    if not (PROJECT_ROOT / path).exists()
]

if missing_paths:
    print("\nFolder yang tidak ditemukan dan akan dilewati:")
    for path in missing_paths:
        print(f" - {path}")

if not existing_paths:
    raise FileNotFoundError(
        "Tidak ada folder output yang tersedia untuk dibuat menjadi handoff."
    )

print("\nFolder yang dimasukkan ke ZIP:")
for path in existing_paths:
    print(f" - {path}")


# ============================================================
# 7. Hapus ZIP dengan nama sama jika kebetulan sudah ada
# ============================================================

if handoff_path.exists():
    handoff_path.unlink()


# ============================================================
# 8. Buat ZIP
#
# Checkpoint besar dikecualikan agar file handoff tetap ringan.
# Hapus pola *.pt, *.pth, *.ckpt dari exclude_patterns apabila
# checkpoint memang perlu ikut dikirim ke Codex.
# ============================================================

exclude_patterns = [
    "*.pt",
    "*.pth",
    "*.ckpt",
    "__pycache__/*",
    "*.pyc",
]

zip_command = [
    "zip",
    "-r",
    str(handoff_path),
    *[str(path) for path in existing_paths],
]

for pattern in exclude_patterns:
    zip_command.extend(["-x", pattern])

subprocess.run(
    zip_command,
    cwd=PROJECT_ROOT,
    check=True,
)


# ============================================================
# 9. Validasi hasil ZIP
# ============================================================

if not handoff_path.exists():
    raise FileNotFoundError(
        f"ZIP gagal dibuat: {handoff_path}"
    )

zip_size_bytes = handoff_path.stat().st_size
zip_size_mb = zip_size_bytes / (1024 ** 2)

zip_test = subprocess.run(
    ["unzip", "-t", str(handoff_path)],
    capture_output=True,
    text=True,
)

if zip_test.returncode != 0:
    raise RuntimeError(
        "ZIP berhasil dibuat, tetapi gagal dalam integrity test.\n"
        + zip_test.stdout
        + "\n"
        + zip_test.stderr
    )


# ============================================================
# 10. Tampilkan ringkasan
# ============================================================

print("\n" + "=" * 60)
print("CODEX HANDOFF BERHASIL DIBUAT")
print("=" * 60)
print(f"Nama file : {handoff_name}")
print(f"Lokasi    : {handoff_path}")
print(f"Ukuran    : {zip_size_mb:.2f} MB")
print(f"Timestamp : {timestamp}")
print(f"Integrity : OK")
print("=" * 60)

print("\nIsi awal ZIP:")
subprocess.run(
    [
        "bash",
        "-lc",
        f'unzip -l "{handoff_path}" | head -80',
    ],
    check=False,
)